In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, 
                             recall_score, f1_score, mean_absolute_error, 
                             mean_squared_error, r2_score, roc_curve, auc)
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

class AswanWeatherAnalysis:
    def __init__(self, filepath):
        """Initialize the analysis with data file"""
        self.df = pd.read_csv(filepath)
        self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        if 'Unnamed: 0' in self.numeric_cols:
            self.numeric_cols.remove('Unnamed: 0')
        
        print("="*80)
        print("ASWAN WEATHER DATA ANALYSIS SYSTEM")
        print("="*80)
        print(f"\nDataset loaded successfully!")
        print(f"Shape: {self.df.shape}")
        print(f"Numeric columns: {self.numeric_cols}")
        
    def data_visualization(self):
        """1. Data Visualization"""
        print("\n" + "="*80)
        print("1. DATA VISUALIZATION")
        print("="*80)
        
        # Distribution plots
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()
        
        for i, col in enumerate(self.numeric_cols[:6]):
            self.df[col].hist(bins=30, ax=axes[i], edgecolor='black')
            axes[i].set_title(f'Distribution of {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequency')
        
        plt.tight_layout()
        plt.savefig('01_distributions.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 01_distributions.png")
        plt.close()
        
        # Time series plot
        plt.figure(figsize=(15, 6))
        for col in ['AvgTemperture', 'Humidity', 'Solar(PV)']:
            plt.plot(self.df.index[:100], self.df[col][:100], label=col, linewidth=2)
        plt.xlabel('Time Index')
        plt.ylabel('Value')
        plt.title('Time Series Analysis (First 100 observations)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig('02_timeseries.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 02_timeseries.png")
        plt.close()
        
    def missing_values_treatment(self):
        """2. Missing Values Treatment"""
        print("\n" + "="*80)
        print("2. MISSING VALUES ANALYSIS")
        print("="*80)
        
        missing = self.df.isnull().sum()
        missing_percent = (missing / len(self.df)) * 100
        missing_df = pd.DataFrame({
            'Column': missing.index,
            'Missing_Count': missing.values,
            'Missing_Percent': missing_percent.values
        })
        missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
        
        if len(missing_df) > 0:
            print("\nMissing Values Summary:")
            print(missing_df.to_string(index=False))
            
            # Visualize missing values
            plt.figure(figsize=(10, 6))
            plt.bar(missing_df['Column'], missing_df['Missing_Count'], color='coral', edgecolor='black')
            plt.xlabel('Columns')
            plt.ylabel('Missing Count')
            plt.title('Missing Values per Column')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.savefig('03_missing_values.png', dpi=300, bbox_inches='tight')
            print("✓ Saved: 03_missing_values.png")
            plt.close()
            
            # Fill missing values with median
            for col in self.numeric_cols:
                if self.df[col].isnull().sum() > 0:
                    self.df[col].fillna(self.df[col].median(), inplace=True)
            print("\n✓ Missing values filled with median")
        else:
            print("\n✓ No missing values found!")
    
    def binning_process(self):
        """3. Binning Process"""
        print("\n" + "="*80)
        print("3. BINNING PROCESS")
        print("="*80)
        
        # Create bins for continuous variables
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()
        
        for i, col in enumerate(self.numeric_cols[:6]):
            # Create 5 bins
            self.df[f'{col}_binned'] = pd.cut(self.df[col], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
            
            # Plot
            self.df[f'{col}_binned'].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
            axes[i].set_title(f'{col} - Binned Distribution')
            axes[i].set_xlabel('Bin')
            axes[i].set_ylabel('Frequency')
            axes[i].tick_params(axis='x', rotation=45)
            
            print(f"\n{col} - Binned Distribution:")
            print(self.df[f'{col}_binned'].value_counts().sort_index())
        
        plt.tight_layout()
        plt.savefig('04_binning.png', dpi=300, bbox_inches='tight')
        print("\n✓ Saved: 04_binning.png")
        plt.close()
    
    def descriptive_statistics(self):
        """4. Descriptive Statistics"""
        print("\n" + "="*80)
        print("4. DESCRIPTIVE STATISTICS")
        print("="*80)
        
        stats_df = pd.DataFrame()
        for col in self.numeric_cols:
            stats_df[col] = {
                'Min': self.df[col].min(),
                'Max': self.df[col].max(),
                'Mean': self.df[col].mean(),
                'Median': self.df[col].median(),
                'Variance': self.df[col].var(),
                'Std': self.df[col].std(),
                'Skewness': self.df[col].skew(),
                'Kurtosis': self.df[col].kurtosis()
            }
        
        stats_df = stats_df.T
        print("\nDescriptive Statistics Summary:")
        print(stats_df.to_string())
        stats_df.to_csv('05_descriptive_stats.csv')
        print("\n✓ Saved: 05_descriptive_stats.csv")
        
        return stats_df
    
    def correlation_analysis(self):
        """5. Correlation and Covariance Analysis"""
        print("\n" + "="*80)
        print("5. CORRELATION & COVARIANCE ANALYSIS")
        print("="*80)
        
        # Covariance Matrix
        cov_matrix = self.df[self.numeric_cols].cov()
        print("\nCovariance Matrix:")
        print(cov_matrix)
        cov_matrix.to_csv('06_covariance_matrix.csv')
        
        # Correlation Matrix
        corr_matrix = self.df[self.numeric_cols].corr()
        print("\nCorrelation Matrix:")
        print(corr_matrix)
        corr_matrix.to_csv('07_correlation_matrix.csv')
        
        # Heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                    square=True, linewidths=1, fmt='.2f', cbar_kws={"shrink": 0.8})
        plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('08_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 08_correlation_heatmap.png")
        plt.close()
        
        return corr_matrix
    
    def statistical_tests(self):
        """6. Statistical Tests (Chi-square, t-test, ANOVA)"""
        print("\n" + "="*80)
        print("6. STATISTICAL TESTS")
        print("="*80)
        
        # T-test: Compare two groups (e.g., high vs low temperature)
        median_temp = self.df['AvgTemperture'].median()
        high_temp_solar = self.df[self.df['AvgTemperture'] > median_temp]['Solar(PV)']
        low_temp_solar = self.df[self.df['AvgTemperture'] <= median_temp]['Solar(PV)']
        
        t_stat, t_pval = stats.ttest_ind(high_temp_solar, low_temp_solar)
        print(f"\nT-Test (Solar PV: High Temp vs Low Temp):")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {t_pval:.4f}")
        print(f"  Result: {'Significant difference' if t_pval < 0.05 else 'No significant difference'}")
        
        # ANOVA: Compare multiple groups
        # Create temperature categories
        self.df['temp_category'] = pd.cut(self.df['AvgTemperture'], bins=3, labels=['Low', 'Medium', 'High'])
        groups = [group['Solar(PV)'].values for name, group in self.df.groupby('temp_category')]
        f_stat, anova_pval = f_oneway(*groups)
        
        print(f"\nANOVA (Solar PV across Temperature Categories):")
        print(f"  F-statistic: {f_stat:.4f}")
        print(f"  p-value: {anova_pval:.4f}")
        print(f"  Result: {'Significant difference' if anova_pval < 0.05 else 'No significant difference'}")
        
        # Z-test for normality
        print(f"\nNormality Test (Shapiro-Wilk) for Solar(PV):")
        w_stat, norm_pval = stats.shapiro(self.df['Solar(PV)'].sample(min(5000, len(self.df))))
        print(f"  W-statistic: {w_stat:.4f}")
        print(f"  p-value: {norm_pval:.4f}")
        print(f"  Result: {'Data is normal' if norm_pval > 0.05 else 'Data is not normal'}")
    
    def pca_analysis(self):
        """7. Principal Component Analysis"""
        print("\n" + "="*80)
        print("7. PRINCIPAL COMPONENT ANALYSIS (PCA)")
        print("="*80)
        
        # Standardize data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.df[self.numeric_cols])
        
        # PCA
        pca = PCA()
        X_pca = pca.fit_transform(X_scaled)
        
        print(f"\nExplained Variance Ratio:")
        for i, var in enumerate(pca.explained_variance_ratio_[:5]):
            print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")
        
        cumsum = np.cumsum(pca.explained_variance_ratio_)
        print(f"\nCumulative Variance Explained:")
        for i in range(min(5, len(cumsum))):
            print(f"  PC1-PC{i+1}: {cumsum[i]:.4f} ({cumsum[i]*100:.2f}%)")
        
        # Scree plot
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(pca.explained_variance_ratio_)+1), 
                pca.explained_variance_ratio_, 'bo-', linewidth=2, markersize=8)
        plt.xlabel('Principal Component')
        plt.ylabel('Explained Variance Ratio')
        plt.title('PCA - Scree Plot')
        plt.grid(True, alpha=0.3)
        plt.savefig('09_pca_scree.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 09_pca_scree.png")
        plt.close()
        
        # PCA scatter
        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=self.df['Solar(PV)'], 
                            cmap='viridis', alpha=0.6, edgecolors='black')
        plt.colorbar(scatter, label='Solar(PV)')
        plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        plt.title('PCA - First Two Principal Components')
        plt.grid(True, alpha=0.3)
        plt.savefig('10_pca_scatter.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 10_pca_scatter.png")
        plt.close()
        
        return X_pca, pca
    
    def lda_analysis(self):
        """8. Linear Discriminant Analysis"""
        print("\n" + "="*80)
        print("8. LINEAR DISCRIMINANT ANALYSIS (LDA)")
        print("="*80)
        
        # Create binary target based on median Solar(PV)
        median_solar = self.df['Solar(PV)'].median()
        y = (self.df['Solar(PV)'] > median_solar).astype(int)
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.df[self.numeric_cols])
        
        # LDA
        lda = LinearDiscriminantAnalysis()
        X_lda = lda.fit_transform(X_scaled, y)
        
        print(f"\nLDA Components Shape: {X_lda.shape}")
        print(f"Explained Variance Ratio: {lda.explained_variance_ratio_}")
        
        # Plot LDA
        plt.figure(figsize=(10, 6))
        plt.scatter(X_lda[y==0, 0], np.zeros(sum(y==0)), alpha=0.6, label='Low Solar', s=50)
        plt.scatter(X_lda[y==1, 0], np.zeros(sum(y==1)), alpha=0.6, label='High Solar', s=50)
        plt.xlabel('LD1')
        plt.title('Linear Discriminant Analysis')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig('11_lda_projection.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 11_lda_projection.png")
        plt.close()
        
        return X_lda, lda
    
    def machine_learning_models(self):
        """9. Machine Learning Models"""
        print("\n" + "="*80)
        print("9. MACHINE LEARNING MODELS")
        print("="*80)
        
        # Prepare data
        X = self.df[self.numeric_cols].drop('Solar(PV)', axis=1, errors='ignore')
        y_reg = self.df['Solar(PV)']
        
        # Create binary classification target
        y_cls = (y_reg > y_reg.median()).astype(int)
        
        # Split data 80-20
        X_train, X_test, y_train_reg, y_test_reg = train_test_split(
            X, y_reg, test_size=0.2, random_state=42)
        _, _, y_train_cls, y_test_cls = train_test_split(
            X, y_cls, test_size=0.2, random_state=42)
        
        print(f"\nTrain size: {len(X_train)} (80%)")
        print(f"Test size: {len(X_test)} (20%)")
        
        # Standardize
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        results = {}
        
        # ===== REGRESSION MODELS =====
        print("\n" + "-"*80)
        print("REGRESSION MODELS (Predicting Solar PV)")
        print("-"*80)
        
        reg_models = {
            'Linear Regression': LinearRegression(),
            'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
            'K-NN (k=5)': KNeighborsRegressor(n_neighbors=5),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(50, 25), max_iter=1000, random_state=42)
        }
        
        for name, model in reg_models.items():
            model.fit(X_train_scaled, y_train_reg)
            y_pred = model.predict(X_test_scaled)
            
            mae = mean_absolute_error(y_test_reg, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
            r2 = r2_score(y_test_reg, y_pred)
            
            # Willmott's Index
            num = np.sum((y_pred - y_test_reg)**2)
            den = np.sum((np.abs(y_pred - y_test_reg.mean()) + np.abs(y_test_reg - y_test_reg.mean()))**2)
            willmott = 1 - (num / den) if den != 0 else 0
            
            # Nash-Sutcliffe Efficiency
            nse = 1 - (np.sum((y_test_reg - y_pred)**2) / np.sum((y_test_reg - y_test_reg.mean())**2))
            
            # Legates-McCabe's Index
            legates = 1 - (np.sum(np.abs(y_test_reg - y_pred)) / np.sum(np.abs(y_test_reg - y_test_reg.mean())))
            
            results[name] = {
                'MAE': mae, 'RMSE': rmse, 'R2': r2,
                'Willmott': willmott, 'NSE': nse, 'Legates': legates
            }
            
            print(f"\n{name}:")
            print(f"  MAE: {mae:.4f}")
            print(f"  RMSE: {rmse:.4f}")
            print(f"  R²: {r2:.4f}")
            print(f"  Willmott's Index: {willmott:.4f}")
            print(f"  Nash-Sutcliffe Efficiency: {nse:.4f}")
            print(f"  Legates-McCabe's Index: {legates:.4f}")
        
        # ===== CLASSIFICATION MODELS =====
        print("\n" + "-"*80)
        print("CLASSIFICATION MODELS (High/Low Solar PV)")
        print("-"*80)
        
        cls_models = {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
            'K-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(50, 25), max_iter=1000, random_state=42)
        }
        
        for name, model in cls_models.items():
            model.fit(X_train_scaled, y_train_cls)
            y_pred = model.predict(X_test_scaled)
            
            # Metrics
            cm = confusion_matrix(y_test_cls, y_pred)
            acc = accuracy_score(y_test_cls, y_pred)
            prec = precision_score(y_test_cls, y_pred, zero_division=0)
            rec = recall_score(y_test_cls, y_pred, zero_division=0)
            f1 = f1_score(y_test_cls, y_pred, zero_division=0)
            error_rate = 1 - acc
            
            results[f"{name} (Classification)"] = {
                'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
                'Error_Rate': error_rate, 'Confusion Matrix': cm
            }
            
            print(f"\n{name}:")
            print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
            print(f"  Error Rate: {error_rate:.4f} ({error_rate*100:.2f}%)")
            print(f"  Precision: {prec:.4f} ({prec*100:.2f}%)")
            print(f"  Recall: {rec:.4f} ({rec*100:.2f}%)")
            print(f"  F1-Score: {f1:.4f} ({f1*100:.2f}%)")
            print(f"\n  Confusion Matrix:")
            print(f"    [[TN={cm[0,0]:<4} FP={cm[0,1]:<4}]")
            print(f"     [FN={cm[1,0]:<4} TP={cm[1,1]:<4}]]")
        
        return results, X_train_scaled, X_test_scaled, y_train_reg, y_test_reg, y_train_cls, y_test_cls
    
    def cross_validation(self, X, y_reg, y_cls):
        """10. K-Fold Cross Validation"""
        print("\n" + "="*80)
        print("10. K-FOLD CROSS VALIDATION (k=5)")
        print("="*80)
        
        kfold = KFold(n_splits=5, shuffle=True, random_state=42)
        
        # Regression
        print("\nRegression Models:")
        lr = LinearRegression()
        scores = cross_val_score(lr, X, y_reg, cv=kfold, scoring='r2')
        print(f"  Linear Regression R² scores: {scores}")
        print(f"  Average R²: {scores.mean():.4f} (+/- {scores.std():.4f})")
        
        # Classification
        print("\nClassification Models:")
        log_reg = LogisticRegression(max_iter=1000, random_state=42)
        scores = cross_val_score(log_reg, X, y_cls, cv=kfold, scoring='accuracy')
        print(f"  Logistic Regression Accuracy scores: {scores}")
        print(f"  Average Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    def visualize_results(self, results):
        """11. Visualize Model Results"""
        print("\n" + "="*80)
        print("11. MODEL COMPARISON VISUALIZATIONS")
        print("="*80)
        
        # Extract regression results
        reg_results = {k: v for k, v in results.items() if 'Classification' not in k}
        
        # Regression metrics comparison
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        metrics = ['MAE', 'RMSE', 'R2', 'Willmott', 'NSE', 'Legates']
        
        for idx, metric in enumerate(metrics):
            ax = axes[idx // 3, idx % 3]
            models = list(reg_results.keys())
            values = [reg_results[m][metric] for m in models]
            
            bars = ax.bar(models, values, color='steelblue', edgecolor='black', alpha=0.7)
            ax.set_ylabel(metric)
            ax.set_title(f'{metric} Comparison')
            ax.tick_params(axis='x', rotation=45)
            
            # Add value labels
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('12_regression_comparison.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 12_regression_comparison.png")
        plt.close()
        
        # Classification metrics comparison
        cls_results = {k: v for k, v in results.items() if 'Classification' in k}
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.ravel()
        cls_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
        
        for idx, metric in enumerate(cls_metrics):
            models = [k.replace(' (Classification)', '') for k in cls_results.keys()]
            values = [cls_results[k][metric] for k in cls_results.keys()]
            
            bars = axes[idx].bar(models, values, color='coral', edgecolor='black', alpha=0.7)
            axes[idx].set_ylabel(metric)
            axes[idx].set_title(f'{metric} Comparison')
            axes[idx].tick_params(axis='x', rotation=45)
            axes[idx].set_ylim([0, 1])
            
            for bar in bars:
                height = bar.get_height()
                axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                              f'{height:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('13_classification_comparison.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 13_classification_comparison.png")
        plt.close()
        
        # Confusion matrices
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        axes = axes.ravel()
        
        for idx, (name, res) in enumerate(cls_results.items()):
            cm = res['Confusion Matrix']
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                       xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
            axes[idx].set_title(f'Confusion Matrix - {name.replace(" (Classification)", "")}')
            axes[idx].set_ylabel('Actual')
            axes[idx].set_xlabel('Predicted')
        
        plt.tight_layout()
        plt.savefig('14_confusion_matrices.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: 14_confusion_matrices.png")
        plt.close()
    
    def print_summary_table(self, results):
        """12. Print Summary Table of All Results"""
        print("\n" + "="*80)
        print("12. COMPREHENSIVE RESULTS SUMMARY")
        print("="*80)
        
        # Regression Summary
        print("\n📊 REGRESSION MODELS SUMMARY:")
        print("-" * 80)
        reg_results = {k: v for k, v in results.items() if 'Classification' not in k}
        
        print(f"{'Model':<25} {'MAE':<10} {'RMSE':<10} {'R²':<10} {'Willmott':<10}")
        print("-" * 80)
        for model, metrics in reg_results.items():
            print(f"{model:<25} {metrics['MAE']:<10.4f} {metrics['RMSE']:<10.4f} "
                  f"{metrics['R2']:<10.4f} {metrics['Willmott']:<10.4f}")
        
        # Classification Summary
        print("\n\n📊 CLASSIFICATION MODELS SUMMARY:")
        print("-" * 80)
        cls_results = {k: v for k, v in results.items() if 'Classification' in k}
        
        print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print("-" * 80)
        for model, metrics in cls_results.items():
            model_name = model.replace(' (Classification)', '')
            print(f"{model_name:<25} {metrics['Accuracy']:<12.4f} {metrics['Precision']:<12.4f} "
                  f"{metrics['Recall']:<12.4f} {metrics['F1']:<12.4f}")
        
        # Best Models
        print("\n\n🏆 BEST PERFORMING MODELS:")
        print("-" * 80)
        
        best_reg_r2 = max(reg_results.items(), key=lambda x: x[1]['R2'])
        best_reg_mae = min(reg_results.items(), key=lambda x: x[1]['MAE'])
        best_cls_acc = max(cls_results.items(), key=lambda x: x[1]['Accuracy'])
        best_cls_f1 = max(cls_results.items(), key=lambda x: x[1]['F1'])
        
        print(f"Best Regression (R²):        {best_reg_r2[0]:<30} (R² = {best_reg_r2[1]['R2']:.4f})")
        print(f"Best Regression (MAE):       {best_reg_mae[0]:<30} (MAE = {best_reg_mae[1]['MAE']:.4f})")
        print(f"Best Classification (Acc):   {best_cls_acc[0].replace(' (Classification)', ''):<30} (Accuracy = {best_cls_acc[1]['Accuracy']:.4f} / {best_cls_acc[1]['Accuracy']*100:.2f}%)")
        print(f"Best Classification (F1):    {best_cls_f1[0].replace(' (Classification)', ''):<30} (F1 = {best_cls_f1[1]['F1']:.4f})")
        
        # Save summary to file
        with open('15_results_summary.txt', 'w') as f:
            f.write("="*80 + "\n")
            f.write("ASWAN WEATHER DATA - COMPREHENSIVE RESULTS SUMMARY\n")
            f.write("="*80 + "\n\n")
            
            f.write("REGRESSION MODELS:\n")
            f.write("-" * 80 + "\n")
            f.write(f"{'Model':<25} {'MAE':<10} {'RMSE':<10} {'R²':<10} {'Willmott':<10}\n")
            f.write("-" * 80 + "\n")
            for model, metrics in reg_results.items():
                f.write(f"{model:<25} {metrics['MAE']:<10.4f} {metrics['RMSE']:<10.4f} "
                       f"{metrics['R2']:<10.4f} {metrics['Willmott']:<10.4f}\n")
            
            f.write("\n\nCLASSIFICATION MODELS:\n")
            f.write("-" * 80 + "\n")
            f.write(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}\n")
            f.write("-" * 80 + "\n")
            for model, metrics in cls_results.items():
                model_name = model.replace(' (Classification)', '')
                f.write(f"{model_name:<25} {metrics['Accuracy']:<12.4f} {metrics['Precision']:<12.4f} "
                       f"{metrics['Recall']:<12.4f} {metrics['F1']:<12.4f}\n")
            
            f.write("\n\nBEST MODELS:\n")
            f.write("-" * 80 + "\n")
            f.write(f"Best Regression (R²):        {best_reg_r2[0]} (R² = {best_reg_r2[1]['R2']:.4f})\n")
            f.write(f"Best Regression (MAE):       {best_reg_mae[0]} (MAE = {best_reg_mae[1]['MAE']:.4f})\n")
            f.write(f"Best Classification (Acc):   {best_cls_acc[0].replace(' (Classification)', '')} (Accuracy = {best_cls_acc[1]['Accuracy']:.4f})\n")
            f.write(f"Best Classification (F1):    {best_cls_f1[0].replace(' (Classification)', '')} (F1 = {best_cls_f1[1]['F1']:.4f})\n")
        
        print("\n✓ Saved: 15_results_summary.txt")
    
    def run_complete_analysis(self):
        """Run all analyses"""
        print("\n\n")
        print("█" * 80)
        print("█" + " " * 78 + "█")
        print("█" + " " * 20 + "STARTING COMPLETE ANALYSIS" + " " * 32 + "█")
        print("█" + " " * 78 + "█")
        print("█" * 80)
        
        # 1. Data Visualization
        self.data_visualization()
        
        # 2. Missing Values
        self.missing_values_treatment()
        
        # 3. Binning
        self.binning_process()
        
        # 4. Descriptive Statistics
        self.descriptive_statistics()
        
        # 5. Correlation Analysis
        self.correlation_analysis()
        
        # 6. Statistical Tests
        self.statistical_tests()
        
        # 7. PCA
        self.pca_analysis()
        
        # 8. LDA
        self.lda_analysis()
        
        # 9. Machine Learning
        results, X_train, X_test, y_train_reg, y_test_reg, y_train_cls, y_test_cls = self.machine_learning_models()
        
        # 10. Cross Validation
        X = self.df[self.numeric_cols].drop('Solar(PV)', axis=1, errors='ignore')
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        y_reg = self.df['Solar(PV)']
        y_cls = (y_reg > y_reg.median()).astype(int)
        self.cross_validation(X_scaled, y_reg, y_cls)
        
        # 11. Visualize Results
        self.visualize_results(results)
        
        # 12. Print Summary Table
        self.print_summary_table(results)
        
        print("\n" + "█" * 80)
        print("█" + " " * 78 + "█")
        print("█" + " " * 24 + "ANALYSIS COMPLETE!" + " " * 33 + "█")
        print("█" + " " * 78 + "█")
        print("█" * 80)
        print("\n✓ All visualizations and results have been saved!")
        print("✓ Check the current directory for PNG files and CSV reports.")


# ===== MAIN EXECUTION =====
if __name__ == "__main__":
    # Initialize analysis
    analyzer = AswanWeatherAnalysis('..\data\AswanData_weatherdata.csv')
    
    # Run complete analysis
    analyzer.run_complete_analysis()

ASWAN WEATHER DATA ANALYSIS SYSTEM

Dataset loaded successfully!
Shape: (398, 8)
Numeric columns: ['AvgTemperture', 'AverageDew(point via humidity)', 'Humidity', 'Wind', 'Pressure', 'Solar(PV)']



████████████████████████████████████████████████████████████████████████████████
█                                                                              █
█                    STARTING COMPLETE ANALYSIS                                █
█                                                                              █
████████████████████████████████████████████████████████████████████████████████

1. DATA VISUALIZATION
✓ Saved: 01_distributions.png
✓ Saved: 02_timeseries.png

2. MISSING VALUES ANALYSIS

✓ No missing values found!

3. BINNING PROCESS

AvgTemperture - Binned Distribution:
AvgTemperture_binned
Very Low      46
Low           60
Medium        63
High         113
Very High    116
Name: count, dtype: int64

AverageDew(point via humidity) - Binned Distribution:
AverageDew(point